# RQ3 — Component Contribution: Which Parts of LLM4Teach Matter?

**Research question:** How much does each architectural component of LLM4Teach contribute to
overall performance? We run ablation experiments that disable or simplify one component at a
time and measure the performance drop relative to the full system.

**Ablation conditions (`small` scenario, primary Qwen-4B teacher, 3 seeds)**

| Condition key | Folder | What is removed / changed |
|---|---|---|
| `full` | `rq3/small/full` | **Full system** — zero-delta baseline |
| `no_history` | `rq3/small/no_history` | History mechanism disabled — LLM sees no past-action context |
| `no_avoidlist` | `rq3/small/no_avoidlist` | Avoid-list disabled — LLM cannot suppress repeatedly failing actions |
| `verbose_prompt` | `rq3/small/verbose_prompt` | Compact prompt replaced with a verbose, unstructured prompt |
| `llm_cached` | `rq3/small/llm_cached` | LLM output cached after first call — no dynamic guidance |

**Data layout:** `runs/{run_tag}/rq3/{scenario}/{condition}/seed{N}/train.csv`

In [ ]:
"""Imports and configuration — RQ3."""
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy import stats

# ── Output directory ──────────────────────────────────────────────────────────
# ── Resolve project root robustly ────────────────────────────────────
import os as _os
try:
    # VS Code injects __vsc_ipynb_file__ — most reliable
    _project_root = Path(__vsc_ipynb_file__).resolve().parent.parent
except NameError:
    # Fallback: walk up from CWD to find the nasim package marker
    _cwd = Path(_os.getcwd()).resolve()
    for _p in [_cwd] + list(_cwd.parents):
        if (_p / "nasim").is_dir():
            _project_root = _p
            break
    else:
        _project_root = _cwd

# -- Output directory --
FIGURES_DIR = _project_root / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUNS_ROOT = _project_root / "runs"

# ── Run + scenario that contain the ablation experiments ──────────────────────
# Ablations live inside a single run folder, on the `small` scenario
# (submit_experiment.sh runs RQ3 for the primary teacher on small).
# Path expected:  runs/{ABLATION_RUN}/rq3/{RQ3_SCENARIO}/{condition}/seed{N}/train.csv
ABLATION_RUN  = "qwen-4B"
RQ3_SCENARIO  = "small"

RQ3_BASE = RUNS_ROOT / ABLATION_RUN / "rq3" / RQ3_SCENARIO

# ── Ablation conditions ───────────────────────────────────────────────────────
FULL_CONDITION = "full"
ABLATIONS      = ["no_history", "no_avoidlist", "verbose_prompt", "llm_cached"]
ALL_CONDITIONS = [FULL_CONDITION] + ABLATIONS

ABLATION_LABELS = {
    "full":           "Full System (baseline)",
    "no_history":     "No History Mechanism",
    "no_avoidlist":   "No Avoid-List",
    "verbose_prompt": "Verbose Prompt",
    "llm_cached":     "LLM Cached (static)",
}

SEEDS = list(range(10))

# ── Colors ────────────────────────────────────────────────────────────────────
FULL_COLOR      = "#0072B2"   # Okabe-Ito blue — full system
NEG_DELTA_COLOR = "#D55E00"   # Okabe-Ito vermillion — ablation hurts
POS_DELTA_COLOR = "#009E73"   # Okabe-Ito green — ablation helps (unusual)

ABLATION_COLORS = {
    "no_history":    "#E69F00",
    "no_avoidlist":  "#D55E00",
    "verbose_prompt":"#CC79A7",
    "llm_cached":    "#999999",
}

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":      150,
    "figure.figsize":  (10, 5),
    "font.size":       12,
    "axes.titlesize":  14,
    "axes.labelsize":  13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.grid":       True,
    "grid.alpha":      0.3,
})

def save_figure(fig, name: str):
    png = FIGURES_DIR / f"{name}.png"
    pdf = FIGURES_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"  Saved: {png}")
    print(f"  Saved: {pdf}")

print("Configuration loaded.")
print(f"  Ablation run : {ABLATION_RUN}")
print(f"  RQ3 base     : {RQ3_BASE}")
print(f"  Conditions   : {ALL_CONDITIONS}")

## 1. Data Discovery & Schema Inspection

In [ ]:
def load_ablation_runs(rq3_base: Path, conditions: list, seeds: list) -> dict:
    """Return {condition: {seed: DataFrame}}."""
    result = {}
    for cond in conditions:
        result[cond] = {}
        for seed in seeds:
            csv = rq3_base / cond / f"seed{seed}" / "train.csv"
            if csv.exists():
                df = pd.read_csv(csv)
                if len(df) > 0:
                    df["seed"]      = seed
                    df["condition"] = cond
                    result[cond][seed] = df
                else:
                    print(f"  [WARN] {csv.relative_to(RUNS_ROOT)} — 0 rows, skipping")
    return result


all_rq3 = load_ablation_runs(RQ3_BASE, ALL_CONDITIONS, SEEDS)

# ── Schema inspection ──────────────────────────────────────────────────────────
print("=" * 70)
print("SCHEMA INSPECTION — train.csv (RQ3 ablations)")
print("=" * 70)
sample = None
for cond in ALL_CONDITIONS:
    sd = all_rq3.get(cond, {})
    if sd:
        sample = next(iter(sd.values()))
        print(f"\nSample: condition={cond}  seed={next(iter(sd))}")
        break

if sample is not None:
    print(f"Shape  : {sample.shape}")
    print("\nColumn dtypes:")
    print(sample.dtypes.to_string())
    print("\nFirst 3 rows:")
    print(sample.drop(columns=["seed","condition"], errors="ignore").head(3).to_string())
    print("\nLast 3 rows:")
    print(sample.drop(columns=["seed","condition"], errors="ignore").tail(3).to_string())
    print(f"\nUnique 'success' values: {sorted(sample['success'].unique())}")

# ── Availability ───────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("AVAILABILITY SUMMARY (RQ3)")
print("=" * 70)
for cond in ALL_CONDITIONS:
    sd          = all_rq3.get(cond, {})
    seeds_found = sorted(sd.keys())
    ep_counts   = [len(sd[s]) for s in seeds_found]
    print(f"  {cond:20s}  seeds={seeds_found}  ep={ep_counts}")
print("=" * 70)

### Column Mapping Notes (RQ3)

| Notebook concept | Actual column | Notes |
|---|---|---|
| Ablation identifier | folder name | Path-encoded; added as `condition` column after loading |
| **Primary metric** | `success` | Binary 0/1 episode success — native task signal, no shaped penalties. Primary panel in Figure 1. |
| Secondary metric | `reward` | Shaped training reward (includes KL and step-penalty terms). Secondary panel only. |
| Final-window performance | last 10 rows | Mean over last 10 episodes per metric |

> **Delta computation:** Δ = metric(ablation) − metric(full system), averaged over the last 10
> episodes. Success-rate delta is the primary panel (left); reward delta is shown alongside as a
> diagnostic. Negative Δ means the ablation hurts; the full-system bar is always at 0.
>
> **⚠ Single-seed limitation:** The current test run has only one seed per ablation condition.
> With one seed, all confidence intervals are zero-width, meaning the chart cannot distinguish a
> real component effect from run-to-run variance. The existing CI machinery will produce proper
> error bars automatically once multiple seeds are available — **no code change needed, only
> more seeds**. Treat single-seed deltas as directional indicators, not definitive claims.

In [ ]:
"""Compute final-performance deltas relative to the full system."""

FINAL_WINDOW = 10

def condition_stats(seed_dict: dict, metric: str, window: int):
    """(mean, ci_half) of the last `window` episodes averaged across seeds."""
    per_seed = [df[metric].tail(window).mean() for df in seed_dict.values()]
    if not per_seed:
        return np.nan, np.nan
    if len(per_seed) == 1:
        return per_seed[0], 0.0
    mu = np.mean(per_seed)
    se = stats.sem(per_seed)
    ci = se * stats.t.ppf(0.975, len(per_seed) - 1)
    return mu, ci


# Gather stats for reward and success
stats_reward  = {}
stats_success = {}
for cond in ALL_CONDITIONS:
    sd = all_rq3.get(cond, {})
    stats_reward[cond]  = condition_stats(sd, "reward",  FINAL_WINDOW)
    stats_success[cond] = condition_stats(sd, "success", FINAL_WINDOW)

full_reward_mu,  full_reward_ci  = stats_reward[FULL_CONDITION]
full_success_mu, full_success_ci = stats_success[FULL_CONDITION]

print(f"Full system — reward : {full_reward_mu:.2f} ± {full_reward_ci:.2f}")
print(f"Full system — success: {full_success_mu:.3f} ± {full_success_ci:.3f}")
print()

# Build delta table for ablations
rows = []
for cond in ABLATIONS:
    r_mu, r_ci   = stats_reward[cond]
    s_mu, s_ci   = stats_success[cond]
    rows.append({
        "condition":      cond,
        "label":          ABLATION_LABELS[cond],
        "delta_reward":   r_mu - full_reward_mu   if not np.isnan(r_mu)  else np.nan,
        "ci_reward":      np.sqrt(r_ci**2 + full_reward_ci**2),
        "delta_success":  s_mu - full_success_mu  if not np.isnan(s_mu)  else np.nan,
        "ci_success":     np.sqrt(s_ci**2 + full_success_ci**2),
        "n_seeds":        len(all_rq3.get(cond, {})),
    })

delta_df = pd.DataFrame(rows)

print("Delta table (ablation − full system):")
print(delta_df[["label","delta_reward","ci_reward","delta_success","ci_success"]].to_string(index=False))

## 2. Figure 1 — Ablation Impact (Horizontal Ordered Bar Chart)

**What this shows:** How much each ablation changes final performance relative to the full
system. Bars are sorted top-to-bottom from most damaging to least — reading top to bottom gives
you a component importance ranking.

**Left panel (primary):** Δ task-success rate — native signal, no shaped penalties.  
**Right panel (secondary):** Δ shaped reward — training diagnostic; shown alongside for
completeness but do not use it to rank components, as KL-penalty magnitudes differ.

> **Note on confidence intervals:** With a single seed per condition the error bars collapse to
> zero — this does not mean effects are certain, it means there are too few observations to
> estimate variance. The bars are directional indicators only until multiple seeds are run.

In [ ]:
"""Figure 1 — Horizontal ordered bar chart: ablation Δ from full system.

LEFT panel (primary)  : Δ task-success rate — native signal, no shaped penalties.
RIGHT panel (secondary): Δ shaped reward    — training diagnostic only.

Rows are sorted by success-rate delta (most damaging on top) so the chart reads as
a component importance ranking.  Reward panel uses the same row order for easy
cross-panel comparison.
"""

def _bar_color(val):
    if np.isnan(val) or abs(val) < 1e-9:
        return "#AAAAAA"
    return NEG_DELTA_COLOR if val < 0 else POS_DELTA_COLOR


# Sort by SUCCESS delta (ascending = most negative on top)
plot_df = delta_df.sort_values("delta_success", ascending=True).reset_index(drop=True)
n_bars  = len(plot_df)
single_seed = delta_df["n_seeds"].max() <= 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(3.5, 0.9 * n_bars + 2.0)))


def _draw_panel(ax, values, errors, labels, title, fmt_fn, xlabel):
    y      = np.arange(len(labels))
    colors = [_bar_color(v) for v in values]
    bars   = ax.barh(y, values, xerr=errors,
                     color=colors, edgecolor="black", linewidth=0.8,
                     capsize=4, height=0.55, zorder=3)
    ax.axvline(0, color="black", linewidth=1.3, zorder=4)

    abs_vals = [abs(v) for v in values if not np.isnan(v)]
    x_pad    = (max(abs_vals) if abs_vals else 1) * 0.04

    for bar, val, err in zip(bars, values, errors):
        if np.isnan(val):
            continue
        x_ann = val + (x_pad if val >= 0 else -x_pad)
        ha    = "left" if val >= 0 else "right"
        txt   = fmt_fn(val)
        if err > 0:
            txt += f" ±{fmt_fn(err).lstrip('+-')}"
        ax.text(x_ann, bar.get_y() + bar.get_height() / 2,
                txt, va="center", ha=ha, fontsize=10.5, fontweight="bold")

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel(xlabel, fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=8)
    ax.grid(True, axis="x", alpha=0.4)
    ax.set_axisbelow(True)


ylabels = plot_df["label"].tolist()

# ── Left: success rate (PRIMARY) ──────────────────────────────────────────────
_draw_panel(
    ax1,
    plot_df["delta_success"].tolist(),
    plot_df["ci_success"].tolist(),
    ylabels,
    title="Δ Task Success Rate  [PRIMARY]",
    fmt_fn=lambda v: f"{v*100:+.1f} pp",
    xlabel="Δ Success Rate vs Full System (percentage points)",
)
ax1.text(0.01, -0.14,
         "← worse than full system          better than full system →",
         transform=ax1.transAxes, fontsize=9, color="#555555", ha="left")

# ── Right: shaped reward (SECONDARY) ─────────────────────────────────────────
_draw_panel(
    ax2,
    plot_df["delta_reward"].tolist(),
    plot_df["ci_reward"].tolist(),
    ylabels,
    title="Δ Shaped Reward  [diagnostic only]",
    fmt_fn=lambda v: f"{v:+.1f}",
    xlabel="Δ Shaped Reward vs Full System",
)
ax2.text(0.01, -0.14,
         "← worse          better →",
         transform=ax2.transAxes, fontsize=9, color="#555555", ha="left")
# Caveat on reward panel
ax2.text(0.99, 0.99,
         "⚠ Shaped reward includes KL penalties\nthat differ across conditions.",
         transform=ax2.transAxes, fontsize=8, color="#8B0000",
         ha="right", va="top",
         bbox=dict(boxstyle="round,pad=0.25", facecolor="#FFF3F3",
                   edgecolor="#D55E00", alpha=0.85))

if single_seed:
    fig.text(0.5, -0.06,
             "⚠ Single seed per condition — zero-width CI is not evidence of certainty. "
             "Add more seeds to obtain valid confidence intervals.",
             ha="center", fontsize=9.5, color="#8B0000",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#FFF3F3",
                       edgecolor="#D55E00", alpha=0.9))

fig.suptitle(
    "RQ3 — Component Contribution: Δ Performance vs Full LLM4Teach System\n"
    "(sorted most → least damaging by success rate; zero line = full system)",
    fontsize=14, fontweight="bold", y=1.04)

plt.tight_layout()
save_figure(fig, "rq3_ablation_impact")
plt.show()

## 3. Figure 2 — Ablation Learning Curves

**What this shows:** Task success rate over training for every ablation condition alongside the
full system. Showing the trajectory (not just the endpoint) reveals whether a component affects
early learning speed, asymptotic performance, or both. The full system is plotted with a thicker
line for easy identification.

> **Reminder:** The curves below are based on a single seed per condition. Apparent
> differences in learning speed or final level may be within normal run-to-run variance. This
> figure is exploratory — add multiple seeds before drawing conclusions about individual
> components.

In [ ]:
"""Figure 2 — Ablation learning curves (task success rate, primary signal).

Metric   : `success` (0/1 per episode), smoothed with a 10-ep rolling mean.
           Success rate is the native task signal — no KL/step-penalty contamination.
Shaped reward curves are available in the debug cell below if needed for diagnostics.
"""

ROLLING_W = 10

ABLATION_COLORS = {
    "no_history":    "#E69F00",   # orange
    "no_avoidlist":  "#D55E00",   # vermillion
    "verbose_prompt":"#CC79A7",   # reddish-purple
    "llm_cached":    "#999999",   # gray
}

fig, ax = plt.subplots(figsize=(12, 6))
single_seed_flag = False

for cond in [FULL_CONDITION] + ABLATIONS:
    sd = all_rq3.get(cond, {})
    if not sd:
        continue

    all_dfs = list(sd.values())
    min_ep  = min(len(df) for df in all_dfs)
    eps     = np.arange(1, min_ep + 1)
    mat     = np.stack([df["success"].values[:min_ep] for df in all_dfs], axis=0)

    col = FULL_COLOR if cond == FULL_CONDITION else ABLATION_COLORS.get(cond, "#AAAAAA")
    lw  = 2.5 if cond == FULL_CONDITION else 1.8
    ls  = "-"  if cond == FULL_CONDITION else "--"
    lbl = ABLATION_LABELS[cond]

    if mat.shape[0] == 1:
        smoothed = pd.Series(mat[0]).rolling(ROLLING_W, min_periods=1, center=True).mean().values
        ax.plot(eps, smoothed * 100, color=col, linestyle=ls, linewidth=lw,
                label=lbl + " (rolling mean)")
        single_seed_flag = True
    else:
        mu = mat.mean(axis=0)
        se = stats.sem(mat, axis=0)
        ci = se * stats.t.ppf(0.975, mat.shape[0] - 1)
        ax.plot(eps, mu * 100, color=col, linestyle=ls, linewidth=lw, label=lbl)
        ax.fill_between(eps, (mu - ci) * 100, (mu + ci) * 100,
                        alpha=0.15, color=col)

ax.set_xlabel("Training Episode", fontsize=13, fontweight="bold")
ax.set_ylabel("Task Success Rate (%)", fontsize=13, fontweight="bold")
ax.set_title("RQ3 — Ablation Learning Curves (success rate)",
             fontsize=15, fontweight="bold", pad=12)
ax.set_ylim([-5, 105])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
ax.legend(loc="lower right", framealpha=0.92)

if single_seed_flag:
    ax.text(0.01, 0.97,
            "Single seed per condition — no CI available.\n"
            "Add seeds before drawing component-level conclusions.",
            transform=ax.transAxes, fontsize=9, color="#8B0000",
            va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFF3F3",
                      edgecolor="#D55E00", alpha=0.9))

plt.tight_layout()
save_figure(fig, "rq3_ablation_curves")
plt.show()

## 4. Export Verification

In [ ]:
expected = [
    "rq3_ablation_impact.png",
    "rq3_ablation_impact.pdf",
    "rq3_ablation_curves.png",
    "rq3_ablation_curves.pdf",
]
print("Export verification:")
all_ok = True
for fname in expected:
    p = FIGURES_DIR / fname
    status = "OK" if p.exists() else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  [{status}] {p}")
print()
if all_ok:
    print("All RQ3 figures exported successfully.")
else:
    print("Some figures are missing — re-run the plotting cells above.")